In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
import pandas as pd
import os as os

df = pd.read_csv("../dataset/dataset_clean.csv")
df = df[df["rating_label"] != "neutral"].copy()
data = df.drop(columns = ["year", "month", "day", "Reviewer Name", "Country", "rating_label"])
labels = df["rating_label"]

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name = "all-MiniLm-L6-v2")

documents = []

for idx, text in enumerate(data["Review Text"]):
    try:
        label = labels.iloc[idx]
    except AttributeError:
        label = labels[idx]
    sentiment_label = "positive" if label == 1 else "negative"

    doc = Document(
        page_content = text,
        metadata ={
            "sentiment" : sentiment_label,
            "id" : idx,
        }
    )

    documents.append(doc)

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory = "../database/chroma_db_reviews"
)

In [ ]:
query = "The room was very dirty and the breakfast was terrible"

results = vector_store.similarity_search(query, k=3)

for i, res in enumerate(results):
    print(res.page_content, res.metadata['sentiment'])

In [ ]:

api_key = open("../venv/api_key.txt", "r", encoding = "utf-8").read()
os.environ["User_Agent"] = "customer-feedback-intelligence"

retriever = vector_store.as_retriever(search_kwargs = {"k" : 5})

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    api_key=api_key,
    temperature=0
)

template = """
Ти — провідний аналітик якості клієнтського сервісу e-commerce платформи.
Твоя задача — відповідати на запитання менеджерів, спираючись ВИКЛЮЧНО на надані відгуки покупців.
Структуруй відповідь: виділи ключові проблеми, причини невдоволення або похвалу від клієнтів.
Якщо у відгуках немає відповіді на запитання, напиши: "Я не знайшов інформації про це у відгуках".
Не вигадуй фактів від себе.

Контекст (відгуки покупців):
{context}

Запитання менеджера: {question}

Аналітичний звіт:
"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

query_1 = "What are the most common complaints regarding delivery speed and package damage?"
print("--- ТЕСТ 1 ---")
print(rag_chain.invoke(query_1))

query_2 = "What do customers say about customer service and refund issues?"
print("\n--- ТЕСТ 2 ---")
print(rag_chain.invoke(query_2))




